# Multi-County Training Workflow - Enhanced

This notebook shows the recommended workflow for training on multiple counties using the new fixes and features.

**New in this version:**
- ✓ Validation step to check data before training
- ✓ Diagnostic analysis to understand class distribution
- ✓ Stratified sampling for balanced class learning
- ✓ Fixed bugs in window generation and class validation

See README_MULTICOUNTY.md, MULTICOUNTY_FIXES.md, and BEFORE_AFTER.md for details.

## Step 1: Setup and Prepare Multi-County Data

In [ ]:
import os
import sys

# Setup paths
os.chdir('/home/sagemaker-user/SageMaker/runoff_classifier')  # Adjust to your path
county_data_dir = 'data/counties'
selected_counties = ['County_A', 'County_B', 'County_C']  # Adjust to your counties

print(f"Working directory: {os.getcwd()}")
print(f"Data directory: {county_data_dir}")
print(f"Selected counties: {selected_counties}")

In [ ]:
# Prepare multi-county training data
from src.utils.prepare_multicounty_training import prepare_multicounty_training

prepare_multicounty_training(
    county_data_dir=county_data_dir,
    selected_counties=selected_counties,
    out_labels='data/merged_labels.gpkg',
    out_vrt='data/merged_mosaic.vrt',
    verified_only=True,
    debug_plots=False,  # Set to True to see overlay plots
)

print("✓ Multi-county data preparation complete")

## Step 2: Validate Merged Data (NEW - Recommended!)

This step validates your merged data before training:
- CRS alignment between raster and labels
- All expected classes are present
- Geometry validity
- Spatial overlap

In [ ]:
import subprocess

result = subprocess.run([
    'python', 'scripts/validate_multicounty.py',
    '--raster_path', 'data/merged_mosaic.vrt',
    '--labels_path', 'data/merged_labels.gpkg',
    '--classes', 'Bank_Erosion', 'Spillway', 'Culvert_Structure', 'Tile_Inlet', 'Tile_Outlet'
])

if result.returncode == 0:
    print("\n✓ Validation passed - data is ready for training")
else:
    print("\n✗ Validation failed - check output above for issues")

## Step 3: Debug and Analyze Training Data (NEW - Diagnostics)

Understand your class distribution and training window generation:

In [ ]:
subprocess.run([
    'python', 'scripts/debug_multicounty.py',
    '--raster_path', 'data/merged_mosaic.vrt',
    '--labels_path', 'data/merged_labels.gpkg',
    '--tile_size', '512'
])

print("\nUse this information to:")
print("  - Identify class imbalance issues")
print("  - Check training window coverage")
print("  - Spot missing classes in certain counties")

## Step 4: Setup Training Parameters

Configure training with the new stratified sampling flag for better multi-county learning.

In [ ]:
# Training configuration
vrt_path = 'data/merged_mosaic.vrt'
labels_path = 'data/merged_labels.gpkg'
train_out_dir = 'outputs/train_multicounty'

os.makedirs(train_out_dir, exist_ok=True)

# Training hyperparameters
config = {
    'task': 'instance_seg',  # or 'detection' for Faster RCNN
    'device': 'cuda',
    'tile_size': 512,
    'epochs': 20,
    'batch_size': 2,
    'grad_accum': 1,
    'lr': 0.0002,
    'num_workers': 8,
    'prefetch_factor': 4,
    'normalize': 'imagenet',
    'max_per_class': 500,
    'stratified_sampling': True,  # NEW: Use stratified sampling for multi-county
}

print("Training Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## Step 5: Run Training with Stratified Sampling (NEW!)

**Key new feature:** `--stratified_sampling` flag ensures balanced class representation during training.

This is especially important for multi-county data where classes may be imbalanced across counties.

In [ ]:
import os

# Setup environment
%env CUDA_VISIBLE_DEVICES=0,1,2,3
%env OMP_NUM_THREADS=4
%env MKL_NUM_THREADS=4
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# Build training command
cmd = f"""python -m torch.distributed.run --nproc_per_node=4 scripts/train.py \
    --task {config['task']} \
    --device {config['device']} \
    --raster_path \"{vrt_path}\" \
    --labels_path \"{labels_path}\" \
    --out_dir \"{train_out_dir}\" \
    --tile_size {config['tile_size']} \
    --epochs {config['epochs']} \
    --batch_size {config['batch_size']} \
    --grad_accum {config['grad_accum']} \
    --lr {config['lr']} \
    --num_workers {config['num_workers']} \
    --prefetch_factor {config['prefetch_factor']} \
    --normalize {config['normalize']} \
    --max_per_class {config['max_per_class']} \
    --stratified_sampling
"""

print("Running training with stratified sampling...")
print(cmd)
print()

os.system(cmd)

## Step 6: Run Inference on Trained Model

In [ ]:
# Find the best checkpoint (or use final model)
import glob

checkpoints = sorted(glob.glob(f"{train_out_dir}/model_epoch_*.pth"))
if checkpoints:
    model_path = checkpoints[-1]  # Use last checkpoint
    print(f"Using model: {model_path}")
else:
    model_path = f"{train_out_dir}/model_final.pth"
    print(f"Using final model: {model_path}")

# Run inference
infer_out_dir = f"{train_out_dir}/inferences"
os.makedirs(infer_out_dir, exist_ok=True)

cmd_infer = f"""python -m torch.distributed.run --nproc_per_node=4 scripts/inference.py \
    --model_path \"{model_path}\" \
    --raster_path \"{vrt_path}\" \
    --out_dir \"{infer_out_dir}\" \
    --tile_size {config['tile_size']} \
    --num_workers 4
"""

print("Running inference...")
print(cmd_infer)
print()

os.system(cmd_infer)

## Step 7: Analyze Results

Check the output detections and compare with baseline (if available).

In [ ]:
import geopandas as gpd
import pandas as pd

# Load inference results
results_path = f"{infer_out_dir}/detections.gpkg"

if os.path.exists(results_path):
    gdf = gpd.read_file(results_path)
    print(f"\nDetection Results:")
    print(f"  Total detections: {len(gdf)}")
    print(f"\nClass distribution:")
    print(gdf['class_name'].value_counts())
    print(f"\nConfidence stats:")
    print(gdf['confidence'].describe())
else:
    print(f"Results file not found: {results_path}")
    print(f"Check inference output directory: {infer_out_dir}")

## Summary of Improvements

This workflow now includes the following fixes and features:

### Bugs Fixed ✓
1. **Variable Shadowing** - Fixed in `src/utils/tiling.py`
   - Reliable training window generation

2. **Missing Class Validation** - Added in `src/data/dataset.py`
   - Early detection of data issues

3. **Unbalanced Sampling** - Fixed with new `StratifiedWeightedSampler`
   - Better multi-county generalization

### New Tools ✓
- `scripts/validate_multicounty.py` - Pre-training data validation
- `scripts/debug_multicounty.py` - Diagnostic analysis
- `src/data/sampling.py` - Stratified sampling implementation

### Expected Results ✓
- More stable training loss
- Better recall for all classes
- Better cross-county generalization
- Detections make sense on multiple counties!

### Documentation ✓
See the following files for details:
- `README_MULTICOUNTY.md` - Quick reference
- `MULTICOUNTY_FIXES.md` - Technical explanation
- `BEFORE_AFTER.md` - Code comparisons
- `IMPLEMENTATION_SUMMARY.md` - What changed
- `INDEX.md` - Complete index